# Experiment 7
## Bagging, Boosting, and Stacked Ensemble Models

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_curve, roc_auc_score

In [ ]:
data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)

print("Shape:", X.shape)
print("Classes:", data.target_names)
X.head()

In [ ]:
X.info()

print("\nMissing values:", X.isnull().sum().sum())
print("\nClass distribution:")
print(y.value_counts())

In [ ]:
plt.figure(figsize=(6,4))
y.value_counts().sort_index().plot(kind="bar")
plt.xticks([0,1], data.target_names, rotation=0)
plt.xlabel("Class")
plt.ylabel("Count")
plt.title("Class Distribution")
plt.show()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

## Bagging

In [ ]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    random_state=42
)

bag_params = {
    "n_estimators": [10, 50, 100],
    "max_samples": [0.5, 0.8, 1.0],
    "max_features": [0.5, 0.8, 1.0]
}

bag_grid = GridSearchCV(
    bag, bag_params, cv=5,
    scoring="accuracy", n_jobs=-1
)

bag_grid.fit(X_train, y_train)

bag_model = bag_grid.best_estimator_

print("Best parameters:", bag_grid.best_params_)
print("Best CV accuracy:", bag_grid.best_score_)

In [ ]:
bag_table = pd.DataFrame(bag_grid.cv_results_)

bag_table = bag_table[
    ["param_n_estimators", "param_max_samples", "param_max_features",
     "mean_test_score", "std_test_score"]
]

bag_table.columns = [
    "n_estimators", "max_samples", "max_features",
    "Avg CV Accuracy", "Std"
]

bag_table["Avg CV Accuracy"] = bag_table["Avg CV Accuracy"] * 100
bag_table = bag_table.sort_values("Avg CV Accuracy", ascending=False)

bag_table.head(10)

In [ ]:
bag_f1 = cross_validate(
    bag_model,
    X_train,
    y_train,
    cv=5,
    scoring="f1"
)

print("Bagging Avg CV F1 Score:", bag_f1["test_score"].mean())

## AdaBoost

In [ ]:
ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1, random_state=42),
    random_state=42
)

ada_params = {
    "n_estimators": [50, 100, 150],
    "learning_rate": [0.01, 0.1, 1.0]
}

ada_grid = GridSearchCV(
    ada, ada_params, cv=5,
    scoring="accuracy", n_jobs=-1
)

ada_grid.fit(X_train, y_train)

ada_model = ada_grid.best_estimator_

print("Best parameters:", ada_grid.best_params_)
print("Best CV accuracy:", ada_grid.best_score_)

In [ ]:
ada_results = pd.DataFrame(ada_grid.cv_results_)

ada_results = ada_results[
    ["param_n_estimators", "param_learning_rate", "mean_test_score"]
]

ada_results.columns = [
    "n_estimators", "learning_rate", "Avg CV Accuracy"
]

ada_results["Avg CV Accuracy"] = ada_results["Avg CV Accuracy"] * 100

ada_f1_scores = []

for _, row in ada_results.iterrows():
    model = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1, random_state=42),
        n_estimators=int(row["n_estimators"]),
        learning_rate=float(row["learning_rate"]),
        random_state=42
    )
    scores = cross_validate(model, X_train, y_train, cv=5, scoring="f1")
    ada_f1_scores.append(scores["test_score"].mean() * 100)

ada_results["Avg CV F1 Score"] = ada_f1_scores

ada_results.sort_values("Avg CV Accuracy", ascending=False)

## Gradient Boosting

In [ ]:
gb = GradientBoostingClassifier(random_state=42)

gb_params = {
    "n_estimators": [50, 100, 150],
    "learning_rate": [0.01, 0.1, 0.2],
    "max_depth": [1, 2, 3]
}

gb_grid = GridSearchCV(
    gb, gb_params, cv=5,
    scoring="accuracy", n_jobs=-1
)

gb_grid.fit(X_train, y_train)

gb_model = gb_grid.best_estimator_

print("Best parameters:", gb_grid.best_params_)
print("Best CV accuracy:", gb_grid.best_score_)

In [ ]:
gb_results = pd.DataFrame(gb_grid.cv_results_)

gb_results = gb_results[
    ["param_n_estimators", "param_learning_rate",
     "param_max_depth", "mean_test_score"]
]

gb_results.columns = [
    "n_estimators", "learning_rate",
    "max_depth", "Avg CV Accuracy"
]

gb_results["Avg CV Accuracy"] = gb_results["Avg CV Accuracy"] * 100

gb_f1_scores = []

for _, row in gb_results.iterrows():
    model = GradientBoostingClassifier(
        n_estimators=int(row["n_estimators"]),
        learning_rate=float(row["learning_rate"]),
        max_depth=int(row["max_depth"]),
        random_state=42
    )
    scores = cross_validate(model, X_train, y_train, cv=5, scoring="f1")
    gb_f1_scores.append(scores["test_score"].mean() * 100)

gb_results["Avg CV F1 Score"] = gb_f1_scores

gb_results.sort_values("Avg CV Accuracy", ascending=False)

## Stacking

In [ ]:
svm = make_pipeline(
    StandardScaler(),
    SVC(probability=True, random_state=42)
)

nb = GaussianNB()

dt = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

lr = LogisticRegression(
    max_iter=1000,
    random_state=42
)

stack = StackingClassifier(
    estimators=[
        ("svm", svm),
        ("nb", nb),
        ("dt", dt)
    ],
    final_estimator=lr,
    cv=5
)

stack_cv = cross_validate(
    stack,
    X_train,
    y_train,
    cv=5,
    scoring=["accuracy", "f1"]
)

print("Base models: SVM, Naive Bayes, Decision Tree")
print("Meta learner: Logistic Regression")
print("Average CV Accuracy:", stack_cv["test_accuracy"].mean())
print("Average CV F1 Score:", stack_cv["test_f1"].mean())

stack.fit(X_train, y_train)

In [ ]:
stack_table = pd.DataFrame({
    "Base Models": ["SVM + Naive Bayes + Decision Tree"],
    "Meta Learner": ["Logistic Regression"],
    "Avg CV Accuracy": [stack_cv["test_accuracy"].mean() * 100],
    "Avg CV F1 Score": [stack_cv["test_f1"].mean() * 100]
})

stack_table

## Performance Comparison

In [ ]:
models = {
    "Bagging": bag_model,
    "AdaBoost": ada_model,
    "Gradient Boosting": gb_model,
    "Stacking": stack
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    results.append([
        name,
        accuracy_score(y_test, pred) * 100,
        precision_score(y_test, pred) * 100,
        recall_score(y_test, pred) * 100,
        f1_score(y_test, pred) * 100
    ])

comparison = pd.DataFrame(
    results,
    columns=["Model", "Accuracy (%)", "Precision (%)", "Recall (%)", "F1 Score (%)"]
)

comparison

## Confusion Matrices

In [ ]:
for name, model in models.items():
    pred = model.predict(X_test)
    print("\n", name)
    print(confusion_matrix(y_test, pred))

In [ ]:
print(classification_report(
    y_test,
    stack.predict(X_test),
    target_names=data.target_names
))

## ROC Curve and AUC

In [ ]:
plt.figure(figsize=(8,6))

for name, model in models.items():
    prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)

    plt.plot(fpr, tpr, label=name + " AUC = " + str(round(auc, 3)))

plt.plot([0,1], [0,1], "--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.grid()
plt.show()

## Observations

In [ ]:
print("Bagging reduces variance by averaging predictions from multiple models.")
print("Boosting reduces bias by learning from errors made by previous models.")
print("AdaBoost gives more importance to difficult or misclassified samples.")
print("Gradient Boosting learns from the errors or residuals of previous models.")
print("Stacking uses different base models and a meta learner to combine their predictions.")
print("The model with the highest test accuracy is:", comparison.loc[comparison["Accuracy (%)"].idxmax(), "Model"])

## Conclusion

Bagging, AdaBoost, Gradient Boosting and Stacking models were implemented using the Wisconsin Diagnostic Breast Cancer dataset. Hyperparameters were evaluated using 5-fold cross-validation and the models were compared using accuracy, precision, recall, F1-score, confusion matrix, ROC curve and AUC. The experiment shows how ensemble methods can improve generalization by reducing variance, reducing bias and combining diverse models.